# System E — Expected Win Rate (EWR) pool ranking (beginner version)

**Goal:** turn a probability engine into a stable leaderboard.

Why you need this:
- ratings are learned from uneven schedules (some fighters fight only a few opponents).
- direct rating comparisons can be sensitive to who fought whom.
- EWR ranks fighters by: *how often would they beat a reference pool?*

---

## 1) Definition

Pick a reference pool $S$ (for example: all active fighters in a weight class).

For each fighter $i$, define:

$$
EWR_i = \frac{1}{|S|}\sum_{j \in S, j\neq i} \Pr(i \text{ beats } j)
$$

- If you have a final stacked probability model $p_{final}(i,j)$, use that.
- Otherwise you can use System A/B/C probabilities directly.

You can also produce a conservative ranking by subtracting uncertainty:
$$
EWR^{LB}_i = EWR_i - z \cdot \text{SE}(EWR_i)
$$

---

## 2) Why it works

EWR is essentially a “soft round-robin”:
- it uses probabilities, not observed head-to-head results,
- it dampens schedule quirks,
- it is easy to compute and cache.

---

## 3) Demo with toy fighters


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

system_e_module = import_module('elo_calculator.application.ranking.system_e_expected_win_rate')
ExpectedWinRatePoolSystem = system_e_module.ExpectedWinRatePoolSystem

In [ ]:
ratings = {
    'fighter_a': 1650.0,
    'fighter_b': 1600.0,
    'fighter_c': 1550.0,
    'fighter_d': 1500.0,
    'fighter_e': 1450.0,
    'fighter_f': 1400.0,
}


def elo_probability(fighter_i: str, fighter_j: str, scale: float = 400.0) -> float:
    rating_diff = ratings[fighter_i] - ratings[fighter_j]
    return 1.0 / (1.0 + 10.0 ** (-rating_diff / scale))


system = ExpectedWinRatePoolSystem()
reference_pool = list(ratings.keys())
rows = system.rank(fighter_ids=reference_pool, reference_pool=reference_pool, probability_fn=elo_probability)

ranking_df = pd.DataFrame(
    [
        {
            'fighter_id': row.fighter_id,
            'rating': ratings[row.fighter_id],
            'ewr': round(row.ewr, 5),
            'adjusted_score': round(row.adjusted_score, 5),
        }
        for row in rows
    ]
)
ranking_df

## 4) Choosing the reference pool in production

Common choices:
- **Active fighters** only (e.g., fought within last 18 months)
- **Weight class specific** pool
- A **top-N** pool for “elite EWR” (e.g., expected win rate vs top 50)

Implementation notes:
- computing all pairwise probabilities is $O(n^2)$; for large pools:
  - restrict to active fighters,
  - compute vs a sampled pool,
  - or compute vs the top-K pool only,
  - cache results per `as_of_date`.

---

## 5) Using EWR with a stacked model

If System D gives you `p_final(i,j)`, EWR becomes:
- your single ranking number that is optimized for predictive accuracy,
- while still producing a stable and interpretable leaderboard.

This is one of the cleanest ways to serve “best possible ranking” from multiple systems.
